# Step-by-Step Guide: Build BLIP from MLIP

# [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dario-coscia/blip/blob/main/notebooks/example.ipynb)

In this tutorial, we’ll walk through how to **transform a Machine Learning Interatomic Potential (MLIP)** into a **Bayesian Learned Interatomic Potential (BLIP)**.

The **BLIP model** is designed to add **Uncertainty Quantification (UQ)** capabilities to message-passing neural networks.  
This not only allows you to estimate prediction confidence, but can also **boost overall performance accuracy** when used during training or fine-tuning.

Let’s get started! 🚀

In [ ]:
! pip install git+https://github.com/dario-coscia/blip.git

In [31]:
# importing the modules

from blip.bayes import BayesianModelWrapper  # the wrapper to covert MLIPs into BLIPs
from blip.posterior import PosteriorModelORB # the Posterior ORB model
from orb_models.forcefield import atomic_system, pretrained  # ORB models, just as work example. Here you can choose your favourite MLIP!
from ase.build import molecule
import torch.nn as nn
import torch

## 🛠 Converting MLIP to BLIP with `BayesianModelWrapper`

To convert any **MLIP** into a **BLIP**, we’ll use the `BayesianModelWrapper`.  
This wrapper is responsible for:

1. **Layer Conversion** – Replacing every `nn.Linear` with a `LinearBayesian` layer  
   *(a linear layer enhanced with the local reparameterization trick)*.
2. **Dropout Injection** – Distributing variational dropout coefficients across the MLIP layers.
3. **KL Divergence Computation** – Adding the KL term for Bayesian regularization.


### 🔍 Step 1 – Import and Inspect an MLIP

As a prototype example, we’ll use the **ORB** model version:

In [6]:
mlip_model = getattr(pretrained, "orb_v3_direct_20_omat")(device="cpu", compile=False)

/Users/dariocoscia/anaconda3/envs/py3.12/lib/python3.12/site-packages/orb_models/utils.py:30: UserWarning: Setting global torch default dtype to torch.float32.
  warnings.warn(f"Setting global torch default dtype to {torch_dtype}.")


You can, of course, choose your own favorite MLIP model instead. We can print the ORB model to inspect it:

In [7]:
print(mlip_model)

DirectForcefieldRegressor(
  (heads): ModuleDict(
    (energy): EnergyHead(
      (normalizer): ScalarNormalizer(
        (bn): BatchNorm1d(1, eps=1e-05, momentum=None, affine=False, track_running_stats=True)
      )
      (mlp): Sequential(
        (NN-0): Linear(in_features=256, out_features=256, bias=True)
        (Act-0): SiLU()
        (NN-1): Linear(in_features=256, out_features=1, bias=True)
        (Act-1): Identity()
      )
      (reference): LinearReferenceEnergy(
        (linear): Linear(in_features=118, out_features=1, bias=False)
      )
    )
    (forces): ForceHead(
      (normalizer): ScalarNormalizer(
        (bn): BatchNorm1d(1, eps=1e-05, momentum=None, affine=False, track_running_stats=True)
      )
      (mlp): Sequential(
        (NN-0): Linear(in_features=256, out_features=256, bias=True)
        (Act-0): SiLU()
        (NN-1): Linear(in_features=256, out_features=3, bias=True)
        (Act-1): Identity()
      )
    )
    (stress): StressHead(
      (diag_norma

The ORB-v3 model architecture can be summarized as follows:

1. **Encoder** → Embeds the input atomic data into a latent representation.
2. **AttentionInteractionNetwork** → Performs message passing using two MLPs:  
   - `_node_mlp`
   - `_edge_mlp`  
   *(There are 5 such attention–interaction blocks.)*
3. **Decoder + Heads** → Produces final predictions for:
   - Energy
   - Forces
   - Stress
   - ... and other properties


## 🔄 Step 2 – Adding BLIP to ORB

We integrate the **BLIP** functionality into the `AttentionInteractionNetwork` by **perturbing the weights** of its MLPs.
This can be interpreted as **injecting Gaussian noise** into the weights, with the noise scaled by the corresponding **variational dropout coefficients**, as explained in the main paper. 

\begin{aligned}
    \mathbf{w}^M_{ls} &= \boldsymbol{\theta}^M_{ls} + \boldsymbol{\epsilon}^M_{ls} \odot |\boldsymbol{\theta}^M_{ls}|, 
    && \boldsymbol{\epsilon}^M_{ls} \sim \mathcal{N}\left(0,\, \alpha_l^n(\mathbf{h}^0, a_{ij})\right), \\
    \mathbf{w}^U_{ls} &= \boldsymbol{\theta}^U_{ls} + \boldsymbol{\epsilon}^U_{ls} \odot |\boldsymbol{\theta}^U_{ls}|, 
    && \boldsymbol{\epsilon}^U_{ls} \sim \mathcal{N}\left(0,\, \alpha^e_l(\mathbf{h}^0)\right),
\end{aligned}

Where:
- $\boldsymbol{\theta}$ → Mean (learned) weights
- $\boldsymbol{\epsilon}$ → Gaussian noise samples
- $\alpha_l^n, \alpha^e_l$ → Variational dropout coefficients  (nodes and edge)
- $\mathbf{h}^0$ → Initial node features  
- $a_{ij}$ → Edge features between atoms *i* and *j*


In essence, BLIP turns deterministic MLP layers inside the message-passing network into **Bayesian layers** that can quantify uncertainty during training and inference. The `AttentionInteractionNetwork` is the following:


In [8]:
print(mlip_model.model.gnn_stacks)

ModuleList(
  (0-4): 5 x AttentionInteractionNetwork(
    (_node_mlp): Sequential(
      (mlp): Sequential(
        (NN-0): Linear(in_features=768, out_features=1024, bias=True)
        (Act-0): SiLU()
        (NN-1): Linear(in_features=1024, out_features=1024, bias=True)
        (Act-1): SiLU()
        (NN-2): Linear(in_features=1024, out_features=256, bias=True)
        (Act-2): Identity()
      )
      (layer_norm): RMSNorm((256,), eps=None, elementwise_affine=True)
    )
    (_edge_mlp): Sequential(
      (mlp): Sequential(
        (NN-0): Linear(in_features=768, out_features=1024, bias=True)
        (Act-0): SiLU()
        (NN-1): Linear(in_features=1024, out_features=1024, bias=True)
        (Act-1): SiLU()
        (NN-2): Linear(in_features=1024, out_features=256, bias=True)
        (Act-2): Identity()
      )
      (layer_norm): RMSNorm((256,), eps=None, elementwise_affine=True)
    )
    (_receive_attn): Linear(in_features=256, out_features=1, bias=True)
    (_send_attn): Line

In order to perturb the weights we use the `BayesianModelWrapper`:

In [ ]:
# get some data (a simple molecule)
atoms = molecule("CH3CH2Cl")
track_data = atomic_system.ase_atoms_to_atom_graphs(atoms, mlip_model.system_config)

# build the Bayesian model mlip -> blip
blip_model = BayesianModelWrapper(model=mlip_model)

# run a warm-up phase
blip_model.warm_up(
    num_nodes=track_data.n_node.sum(),
    num_edges=track_data.n_edge.sum(),
    batch=track_data,
    regex_pattern=[
        r"^model\.gnn_stacks\.\d+\._(node_mlp|edge_mlp)\.mlp\.NN-\d+$"
    ],  # here you should insert a regex explaining which layers need to be convert from Linear to LinearBayesian
)

Done! It's super easy!
Before training, we run a **warm-up phase** to initialize the internal modules.  
This step prepares the model for **scattering the posterior variational dropout coefficients** into the appropriate layers. When running the warm-up, you must specify:

1. **A `batch` of data** – A representative mini-batch from your dataset.
2. **Number of nodes** – The number of atoms (or graph nodes) in the batch.
3. **Number of edges** – The number of pairwise interactions in the batch.
4. **Regex pattern** – A regular expression that matches the layer names you want to convert from `Linear` to `LinearBayesian`.

Now we can again inspect the `gnn_stack` and observed that the `Linear` layers have been transformed to `LinearBayesian` layers

In [12]:
print(blip_model.model.model.gnn_stacks)

ModuleList(
  (0-4): 5 x AttentionInteractionNetwork(
    (_node_mlp): Sequential(
      (mlp): Sequential(
        (NN-0): LinearBayesian(in_features=768, out_features=1024, bias=True)
        (Act-0): SiLU()
        (NN-1): LinearBayesian(in_features=1024, out_features=1024, bias=True)
        (Act-1): SiLU()
        (NN-2): LinearBayesian(in_features=1024, out_features=256, bias=True)
        (Act-2): Identity()
      )
      (layer_norm): RMSNorm((256,), eps=None, elementwise_affine=True)
    )
    (_edge_mlp): Sequential(
      (mlp): Sequential(
        (NN-0): LinearBayesian(in_features=768, out_features=1024, bias=True)
        (Act-0): SiLU()
        (NN-1): LinearBayesian(in_features=1024, out_features=1024, bias=True)
        (Act-1): SiLU()
        (NN-2): LinearBayesian(in_features=1024, out_features=256, bias=True)
        (Act-2): Identity()
      )
      (layer_norm): RMSNorm((256,), eps=None, elementwise_affine=True)
    )
    (_receive_attn): Linear(in_features=256, o

## 📈 Step 3 – Training and Inference with BLIP

To train a BLIP model, you must define a **posterior network**.  

This can be done either:
- at initialization via `BayesianModelWrapper(..., posterior=)`, or  
- later using `.add_posterior_net(...)` attribute.


In [22]:
posterior = PosteriorModelORB(
    node_dim=track_data.node_features["atomic_numbers_embedding"].shape[-1],
    edge_dim=23,
    num_alphas_edge=blip_model.num_alphas_edge,  # number of dropout coefficients on edges
    num_alphas_node=blip_model.num_alphas_node,  # number of dropout coefficients on nodes
    hidden_dim=64,
    n_node_layers=2,
    n_edge_layers=2,
    activation=nn.SiLU,
)

blip_model.add_posterior_net(posterior)

With the warm-up complete ✅ and the posterior net added ✅,   you now have all the ingredients for finetuing the BLIP model.
You can perform a forward pass by:

In [ ]:
blip_model(batch=track_data) # use always kwargs

{'node_features': tensor([[-3.3321,  0.0854, -0.2931,  ...,  0.9112,  3.0176, -3.5071],
         [ 2.3280,  0.4152, -0.2751,  ..., -2.9178, -0.6304,  1.2753],
         [-2.4561,  1.4464,  0.0323,  ..., -5.1443,  1.2651,  2.0157],
         ...,
         [-0.7367,  2.2291,  0.0303,  ..., -3.5324, -0.2649, -1.8156],
         [ 0.7697, -0.8735,  0.4278,  ...,  2.0982, -0.5379, -0.6768],
         [ 2.2515,  1.0811,  0.9414,  ..., -2.5110, -1.0546,  0.1774]],
        grad_fn=<AddBackward0>),
 'edge_features': tensor([[ -2.6911,  -4.7756,  -5.3994,  ...,  -2.9721,   1.2422,   1.5989],
         [ -2.2226,  -8.1334,  -5.4546,  ...,  -1.1810,  -0.8763,  -0.3908],
         [  0.1985,  -9.4125,   3.3782,  ...,  -1.3492,   1.7524,  -2.8808],
         ...,
         [  1.7528,  32.5968,  13.4493,  ...,  -2.9326,  -0.2181,   3.1338],
         [ -0.3301,  -2.8093,  -0.7381,  ...,  -1.9070,   1.8147,   1.5666],
         [  2.0194, -13.8108,  -1.4831,  ...,  -1.2934,   2.9843,  -2.7953]],
        grad_fn

⚠️ If you want the **MAP** (Maximum A Posteriori) estimate, simply set the model to eval mode. This disables weight sampling — you’ll get deterministic predictions and **no** uncertainty quantification.


In [29]:
blip_model.eval()
blip_model(batch=track_data)["node_features"] == mlip_model(track_data)["node_features"]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

Since we haven’t fine-tuned the model with BLIP yet, the **MAP prediction** of the BLIP is **identical** to the original MLIP. After fine-tuning, BLIP may produce different MAP estimates and also provide uncertainty information. Finally, you can train as a standard `nn.Module`, but in the loss function you will need to add the KL penalty, which can be computed easily by:

In [34]:
edge_index = torch.stack([track_data.senders, track_data.receivers], dim=0)
kl = blip_model.kl_loss(edge_index=edge_index)
print(kl)

tensor(0.5675, grad_fn=<AddBackward0>)


## 🎉 Conclusion

And that’s it! ✅  

With `BayesianModelWrapper`, you can:
- Train a Bayesian version of any MLIP without the need to rewrite from scratch the main logics
- Perform stochastic inference to obtain multiple samples or MAP estimate to mean prediction

Because BLIP is stochastic by default, once trained you can query it multiple times to estimate predictive distributions and quantify uncertainty in your predictions. For more details see the main article, and visit the main repository for experiment examples.
